# Matrix multiplication in AI

In [ ]:
import numpy as np

random_generator = np.random.default_rng(0)

## Counting the operations

Using the FLOP count from notebook 02, compare the work of each part of one realistic layer (4,096 inputs, 4,096 neurons, a batch of 32 examples):

- the matrix multiplication: $2 \times \text{batch} \times \text{inputs} \times \text{neurons}$;
- adding the bias and applying ReLU: one operation per output value each.

In [ ]:
def dense_layer_flops(batch_size: int, input_count: int, neuron_count: int) -> dict[str, int]:
    return {
        "matrix multiplication": 2 * batch_size * input_count * neuron_count,
        "bias": batch_size * neuron_count,
        "relu": batch_size * neuron_count,
    }

layer_flops = dense_layer_flops(batch_size=32, input_count=4096, neuron_count=4096)
layer_flops

{'matrix multiplication': 1073741824, 'bias': 131072, 'relu': 131072}

In [ ]:
f"{layer_flops['matrix multiplication'] / sum(layer_flops.values()):.2%} of the work is the matrix multiplication"

'99.98% of the work is the matrix multiplication'

The matrix multiplication's cost multiplies *three* sizes together; everything else multiplies only two. The wider the layer, the more completely the matrix multiplication dominates.

## Images: filters are dot products

Models for images, called **convolutional neural networks** (CNNs), use **filters**: small grids of weights, for example 3 × 3, that slide over the image. At every position, the filter computes the dot product between its weights and the patch of pixels underneath. The result is large where the patch looks like the pattern the filter encodes.

This image is dark on the left and bright on the right, and the filter below detects exactly that kind of vertical edge:

In [ ]:
image = np.zeros((6, 6))
image[:, 3:] = 9.0

vertical_edge_filter = np.array([
    [-1.0, 0.0, 1.0],
    [-1.0, 0.0, 1.0],
    [-1.0, 0.0, 1.0],
])
image

array([[0., 0., 0., 9., 9., 9.],
       [0., 0., 0., 9., 9., 9.],
       [0., 0., 0., 9., 9., 9.],
       [0., 0., 0., 9., 9., 9.],
       [0., 0., 0., 9., 9., 9.],
       [0., 0., 0., 9., 9., 9.]])

In [ ]:
filter_size = 3
output_size = image.shape[0] - filter_size + 1
filter_response = np.zeros((output_size, output_size))
for row in range(output_size):
    for column in range(output_size):
        patch = image[row:row + filter_size, column:column + filter_size]
        filter_response[row, column] = patch.reshape(-1) @ vertical_edge_filter.reshape(-1)

filter_response

array([[ 0., 27., 27.,  0.],
       [ 0., 27., 27.,  0.],
       [ 0., 27., 27.,  0.],
       [ 0., 27., 27.,  0.]])

The response is high only in the columns where the edge is. A CNN learns thousands of such filters from data, and applying them means computing **huge numbers of dot products**. Libraries reorganise all of them into a single large matrix multiplication (next section).

## Convolution as one matrix multiplication

The **im2col** trick ("image to columns") copies every patch the filter will visit into one row of a matrix. The whole filtering operation then becomes a single matrix multiplication between that patch matrix and the flattened filter. It uses extra memory for the copies, but it lets the library reuse its fastest routine. It is one of several strategies libraries choose from; others rely on transforms such as the FFT.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

# Each row of patch_matrix is one 3x3 patch flattened into 9 values
patch_matrix = sliding_window_view(image, (filter_size, filter_size)).reshape(-1, filter_size * filter_size)
patch_matrix.shape

(16, 9)

In [ ]:
response_as_matmul = (patch_matrix @ vertical_edge_filter.reshape(-1)).reshape(output_size, output_size)
np.allclose(response_as_matmul, filter_response)

True

## Transformers: the models behind chatbots

Notebook 02 introduced Transformers in words. Here are the same ideas as small computations, with toy sizes.

### Text becomes a matrix

The text is split into **tokens**, and each token is replaced by a vector of numbers taken from a table that the model learns during training. That vector is called an **embedding**. So a sentence becomes a matrix with one row per token, exactly like the sentence tensors of notebook 02. (This toy example uses whole words and 4 numbers per token; real models use pieces of words and thousands of numbers.)

In [ ]:
vocabulary = {"the": 0, "bank": 1, "of": 2, "river": 3, "approved": 4, "loan": 5}
embedding_table = random_generator.standard_normal((len(vocabulary), 4))

sentence = ["the", "bank", "of", "the", "river"]
token_ids = [vocabulary[word] for word in sentence]
sentence_matrix = embedding_table[token_ids]
token_ids, sentence_matrix.shape

([0, 1, 2, 0, 3], (5, 4))

### Attention: every token looks at the others

Words need context. "Bank" means something different in *"the bank of the river"* and in *"the bank approved the loan"*. **Attention** is the mechanism that lets each token look at every other token in the text, decide how relevant each one is, and mix in information from the relevant ones. In the first sentence, "bank" would pay attention to "river".

How relevant one token is to another is measured with a **dot product** between their vectors: vectors that point in similar directions give large dot products. Computing it for **every pair** of tokens at once is a single matrix multiplication of the sentence matrix with its own transpose ($S S^\top$, where the transpose swaps rows and columns):

In [ ]:
relevance_scores = sentence_matrix @ sentence_matrix.T
relevance_scores.shape

(5, 5)

One score for each of the 5 × 5 pairs of tokens. A real Transformer first transforms the vectors with learned weight matrices (more matrix multiplications), turns the scores into proportions, and uses another matrix multiplication to mix the vectors. The principle is the same.

### Everything else is layers

After attention, every token passes through ordinary layers like the ones at the start of this notebook, and the whole block is repeated dozens of times. Almost every parameter of an LLM lives in a weight matrix.

This gives a useful rule of thumb: producing one token costs about **2 FLOPs per parameter**, one multiplication and one addition for each weight. (It ignores the attention scores, whose cost grows with the length of the text, so it is an approximation.)

In [ ]:
parameter_count = 70_000_000_000
flops_per_token = 2 * parameter_count
answer_tokens = 650
f"{flops_per_token:.1e} FLOPs per token, {flops_per_token * answer_tokens:.1e} FLOPs for a ~500-word answer"

'1.4e+11 FLOPs per token, 9.1e+13 FLOPs for a ~500-word answer'

About a hundred trillion operations for one medium-length answer from a 70-billion-parameter model, nearly all of them inside matrix multiplications. The last section of this notebook measures this directly on a real Transformer block.

## Batches of matrix multiplications

When `@` receives tensors with more than two dimensions, it treats the last two as matrices and multiplies them **in pairs** along the other dimensions. This *batched* matrix multiplication lets AI libraries process many examples, or many parts of a model, in a single call.

In [ ]:
random_generator = np.random.default_rng(0)
first_stack = random_generator.standard_normal((10, 4, 3))
second_stack = random_generator.standard_normal((10, 3, 2))
batched_product = first_stack @ second_stack
batched_product.shape

(10, 4, 2)

Each of the ten 4 × 2 results is an ordinary matrix product of the matching pair:

In [ ]:
np.allclose(batched_product[7], first_stack[7] @ second_stack[7])

True

## Measuring a real Transformer block

A **profiler** is a tool that records how much time a program spends in each of its operations. PyTorch, the most widely used deep-learning library, includes one. We run a standard Transformer encoder block (the building block of models such as BERT) on the CPU, in inference mode.

In [ ]:
import torch
from torch.profiler import ProfilerActivity, profile

encoder_block = torch.nn.TransformerEncoderLayer(d_model=512, nhead=8, dim_feedforward=2048, batch_first=True)
encoder_block.eval()
token_batch = torch.randn(32, 128, 512)

with torch.no_grad():
    # Warm-up run so one-off initialisation costs are not profiled
    encoder_block(token_batch)
    with profile(activities=[ProfilerActivity.CPU]) as profiler:
        encoder_block(token_batch)

USDT:2026-09-27 20:43:10 987677:987677 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-27 20:43:10 987677:987677 SyncActivityProfilerHandler.cpp:46] profiler_stop


(The two `USDT` lines are harmless log messages printed by this PyTorch build.)

Each entry below is one internal PyTorch operation, with its share of the total time. *Self* time excludes time spent in operations it calls, so the shares can be added up safely.

In [ ]:
operation_times = {event.key: event.self_cpu_time_total for event in profiler.key_averages()}
total_time = sum(operation_times.values())
top_operations = sorted(operation_times.items(), key=lambda item: item[1], reverse=True)[:8]
{name: f"{time / total_time:.1%}" for name, time in top_operations}

{'aten::addmm': '27.7%',
 'aten::mm': '27.6%',
 'aten::_addmm_activation': '21.4%',
 'aten::copy_': '5.4%',
 'aten::bmm': '5.0%',
 'aten::_transformer_encoder_layer_fwd': '4.1%',
 'aten::_transform_bias_rescale_qkv': '3.8%',
 'aten::add_': '1.3%'}

`aten::addmm` is "add + matrix multiply" (a layer, $XW + b$), `aten::_addmm_activation` is the same fused with ReLU, `aten::mm` is a plain matrix multiplication and `aten::bmm` a batched one (see the section on batches above):

In [ ]:
matmul_operations = {"aten::addmm", "aten::_addmm_activation", "aten::mm", "aten::bmm", "aten::matmul"}
matmul_time = sum(time for name, time in operation_times.items() if name in matmul_operations)
f"{matmul_time / total_time:.1%} of the time is spent multiplying matrices"

'81.7% of the time is spent multiplying matrices'

## Key ideas

- In a wide layer, the matrix multiplication accounts for almost all the arithmetic; bias and activation are negligible.
- Image **filters** compute dot products with every patch; the **im2col** trick turns all of them into one matrix multiplication.
- In a Transformer, token embeddings form a matrix, and the relevance of every pair of tokens is one matrix multiplication.
- A **profiler** confirms it on a real Transformer block: most of the time is spent in matrix multiplication operations.